# 02 - Negative examples - data not at risk
Datacite

In [ ]:
# Imports

import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

import time, re, requests
from general.util.banned_words import add_flagged_column

import pandas as pd
from pathlib import Path
import ast
import numpy as np

ALL

In [ ]:
MAILTO = "maja.murawka@student.uva.nl"
session = requests.Session()
session.headers.update({"Accept":"application/vnd.api+json","User-Agent":f"metadata-harvest/1.0 (mailto:{MAILTO})"})

FEDERAL_CLIENT_IDS = [
    "climsc.csc","dhs.dhs","doe.bnl","doe.inl","doe.lbnl","doe.netl","doe.nrel","doe.ornl","doe.osti","doe.pnnl","dot.dot",
    "epa.epa","esdis.eosdis","esdis.ornl","fwri.repo","gshs.gshs","heliophy.spdf","nasa.ahed","nasa.data","nasa.genelab",
    "nasaarmd.afpda","nasapds.nasapds","nasasmd.pgda","nasasmd.psi","nasasmd.shadoz","nihnci.canano","nihnci.nccr",
    "nihnci.nci","nihnci.tcia","nist.admin","nist.pdr","noaa.gmd","noaa.library","noaa.ncei","noaa.ncei-ncl","pryl.igsn",
    "pryl.mxfyrs","pryl.nsf-ice","si.ccrcn","si.cda","si.emammal","si.figshare","si.si","usgs.fws","usgs.fws-lcc",
    "usgs.prod","usnps.usnps","xaqp.zqnehk"
]

RICH_DESC_TYPES = {"abstract", "methods", "technicalinfo", "other", "tableofcontents", "seriesinformation"}

def load_banned_words_union(base: Path):
    with open((base / "../general/data/banned_words_PEN.txt").resolve(), encoding="utf-8") as f:
        pen_words = sorted(set(line.strip().casefold() for line in f if line.strip() and not line.startswith("#")))
    with open((base / "../general/data/banned_words.txt").resolve(), encoding="utf-8") as f:
        nyt_words = sorted(set(line.strip().casefold() for line in f if line.strip() and not line.startswith("#")))
    return sorted(set(pen_words) | set(nyt_words))

def build_fast_patterns(words):
    singles = [w for w in words if " " not in w]
    phrases = [w for w in words if " " in w]
    single_pat = re.compile(r"(?i)\b(" + "|".join(map(re.escape, singles)) + r")\b") if singles else None
    phrase_pat = re.compile(r"(?i)(" + "|".join(map(re.escape, phrases)) + r")") if phrases else None
    return single_pat, phrase_pat

def fast_has_any(text, single_pat, phrase_pat):
    if not isinstance(text, str) or not text.strip():
        return False
    return (single_pat and single_pat.search(text) is not None) or (phrase_pat and phrase_pat.search(text) is not None)

def has_us_affiliation(creators_or_contributors):
    for person in creators_or_contributors or []:
        for aff in (person.get("affiliation") or []):
            if not isinstance(aff, dict):
                continue
            name = (aff.get("name") or "").casefold()
            if "united states" in name or "u.s." in name or " usa" in name or name.endswith(", us") or name.endswith(" us"):
                return True
    return False

def word_count(s):
    return len(re.findall(r"\b\w+\b", s)) if isinstance(s, str) else 0

def extract_title(attrs):
    titles = attrs.get("titles") or []
    for t in titles:
        txt = t.get("title")
        if isinstance(txt, str) and txt.strip():
            return txt
    return None

def extract_rich_descriptions(attrs):
    out = {f"dc_desc_{t}": None for t in sorted(RICH_DESC_TYPES)}
    descs = attrs.get("descriptions") or []

    for d in descs:
        dt = (d.get("descriptionType") or "").strip().casefold()
        txt = d.get("description")
        if dt in RICH_DESC_TYPES and isinstance(txt, str) and txt.strip():
            col = f"dc_desc_{dt}"
            if out[col] is None or word_count(txt) > word_count(out[col]):
                out[col] = txt

    best_txt, best_type, best_wc = None, None, 0
    for dt in ["abstract","methods","technicalinfo","other","tableofcontents","seriesinformation"]:
        txt = out.get(f"dc_desc_{dt}")
        wc = word_count(txt)
        if wc > 0 and (best_txt is None or (dt in ["abstract","methods","technicalinfo"] and best_type not in ["abstract","methods","technicalinfo"]) or wc > best_wc):
            best_txt, best_type, best_wc = txt, dt, wc

    out["dc_text_best"] = best_txt
    out["dc_text_best_type"] = best_type
    out["dc_text_best_wc"] = best_wc
    return out

def chunk(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def datacite_cursor_page(query, resource_type_id="dataset", client_ids=None, page_size=1000, cursor="1"):
    params = {
        "disable-facets":"true",
        "query":query,
        "resource-type-id":resource_type_id,
        "page[size]":int(page_size),
        "page[cursor]":cursor,
        "affiliation":"true",
        "publisher":"true"
    }
    if client_ids:
        params["client-id"] = ",".join(client_ids)

    r = session.get("https://api.datacite.org/dois", params=params, timeout=90)
    if r.status_code == 429:
        time.sleep(2)
        r = session.get("https://api.datacite.org/dois", params=params, timeout=90)
    r.raise_for_status()

    js = r.json()
    data = js.get("data") or []
    next_url = (js.get("links") or {}).get("next")

    return data, next_url

def parse_cursor_from_next(next_url):
    if not next_url:
        return None
    m = re.search(r"page%5Bcursor%5D=([^&]+)", next_url) or re.search(r"page\[cursor\]=([^&]+)", next_url)
    return m.group(1) if m else None

def make_rich_types_query():
    # DataCite supports filtering on descriptionType tokens like Abstract/Methods/TechnicalInfo/Other/TableOfContents/SeriesInformation
    return "(" + " OR ".join([
        "descriptions.descriptionType:Abstract",
        "descriptions.descriptionType:Methods",
        "descriptions.descriptionType:TechnicalInfo",
        "descriptions.descriptionType:Other",
        "descriptions.descriptionType:TableOfContents",
        "descriptions.descriptionType:SeriesInformation",
    ]) + ")"

RICH_TYPES_QUERY = make_rich_types_query()

def scrape_B_federal_no_flagged_cursor(
    sample_size,
    banned_union,
    page_size=1000,
    max_pages_per_clientchunk=20,
    client_chunk_size=30,
    query_base="language:en",
    resource_type_id="dataset"
):
    single_pat, phrase_pat = build_fast_patterns(banned_union)
    rows, seen = [], set()
    drops = {"dupe":0, "no_rich_text":0, "has_kw":0}

    q = f"({query_base}) AND {RICH_TYPES_QUERY}"

    client_chunks = list(chunk(FEDERAL_CLIENT_IDS, client_chunk_size))
    for ci, client_ids in enumerate(client_chunks, start=1):
        if len(rows) >= sample_size:
            break

        cursor = "1"
        for p in range(1, max_pages_per_clientchunk + 1):
            if len(rows) >= sample_size:
                break

            data, next_url = datacite_cursor_page(
                query=q,
                resource_type_id=resource_type_id,
                client_ids=client_ids,
                page_size=page_size,
                cursor=cursor
            )

            added = 0
            for item in data:
                if len(rows) >= sample_size:
                    break

                attrs = item.get("attributes") or {}
                rel = item.get("relationships") or {}

                doi = (attrs.get("doi") or item.get("id") or "").strip().lower()
                if not doi or doi in seen:
                    drops["dupe"] += 1
                    continue

                client_id = (((rel.get("client") or {}).get("data")) or {}).get("id")
                client_id = (client_id or "").strip().lower() if client_id else None

                rich = extract_rich_descriptions(attrs)
                if not rich.get("dc_text_best"):
                    drops["no_rich_text"] += 1
                    continue

                if fast_has_any(rich["dc_text_best"], single_pat, phrase_pat):
                    drops["has_kw"] += 1
                    continue

                row = {
                    "dataset_group": "B",
                    "doi": doi,
                    "repository_code": client_id,
                    "dc_language": attrs.get("language"),
                    "dc_publicationYear": attrs.get("publicationYear"),
                    "dc_url": attrs.get("url"),
                    "dc_title": extract_title(attrs),
                    "dc_types_resourceTypeGeneral": (attrs.get("types") or {}).get("resourceTypeGeneral"),
                    "has_any_keyword": 0,
                    "dc_publisher": attrs.get("publisher"),
                    "dc_fundingReferences": attrs.get("fundingReferences"),
                }
                row.update(rich)

                rows.append(row)
                seen.add(doi)
                added += 1

            print(f"[B_CURSOR] clients_chunk={ci}/{len(client_chunks)} page={p} added={added} total={len(rows)}/{sample_size} drops={drops}")

            cursor = parse_cursor_from_next(next_url)
            if not cursor:
                break

    return pd.DataFrame(rows).head(sample_size)

def make_keyword_query(words_chunk):
    parts = [f'descriptions.description:"{w}"' for w in words_chunk]
    return f"(language:en) AND {RICH_TYPES_QUERY} AND (" + " OR ".join(parts) + ")"

def scrape_C_flagged_no_us_aff_cursor(
    sample_size,
    banned_union,
    page_size=1000,
    max_pages=200,
    kw_chunk_size=20,
    query_base="language:en",
    resource_type_id="dataset"
):
    single_pat, phrase_pat = build_fast_patterns(banned_union)
    rows, seen = [], set()
    drops = {"dupe":0, "no_rich_text":0, "no_kw":0, "us_aff":0}

    kw_chunks = list(chunk(banned_union, kw_chunk_size))

    for qi, words_chunk in enumerate(kw_chunks, start=1):
        if len(rows) >= sample_size:
            break

        q = make_keyword_query(words_chunk)
        if query_base:
            q = f"({query_base}) AND {q}"

        cursor = "1"
        for p in range(1, max_pages + 1):
            if len(rows) >= sample_size:
                break

            data, next_url = datacite_cursor_page(
                query=q,
                resource_type_id=resource_type_id,
                client_ids=None,
                page_size=page_size,
                cursor=cursor
            )

            added = 0
            for item in data:
                if len(rows) >= sample_size:
                    break

                attrs = item.get("attributes") or {}
                rel = item.get("relationships") or {}

                doi = (attrs.get("doi") or item.get("id") or "").strip().lower()
                if not doi or doi in seen:
                    drops["dupe"] += 1
                    continue

                rich = extract_rich_descriptions(attrs)
                if not rich.get("dc_text_best"):
                    drops["no_rich_text"] += 1
                    continue

                if not fast_has_any(rich["dc_text_best"], single_pat, phrase_pat):
                    drops["no_kw"] += 1
                    continue

                creators = attrs.get("creators") or []
                contributors = attrs.get("contributors") or []
                if has_us_affiliation(creators) or has_us_affiliation(contributors):
                    drops["us_aff"] += 1
                    continue

                client_id = (((rel.get("client") or {}).get("data")) or {}).get("id")
                client_id = (client_id or "").strip().lower() if client_id else None

                row = {
                    "dataset_group": "C",
                    "doi": doi,
                    "repository_code": client_id,
                    "dc_language": attrs.get("language"),
                    "dc_publicationYear": attrs.get("publicationYear"),
                    "dc_url": attrs.get("url"),
                    "dc_title": extract_title(attrs),
                    "dc_types_resourceTypeGeneral": (attrs.get("types") or {}).get("resourceTypeGeneral"),
                    "has_us_affiliation": 0,
                    "has_any_keyword": 1,
                    "dc_publisher": attrs.get("publisher"),
                    "dc_fundingReferences": attrs.get("fundingReferences"),
                }
                row.update(rich)

                rows.append(row)
                seen.add(doi)
                added += 1

            print(f"[C_CURSOR] kw_chunk={qi}/{len(kw_chunks)} page={p} added={added} total={len(rows)}/{sample_size} drops={drops}")

            cursor = parse_cursor_from_next(next_url)
            if not cursor:
                break

    return pd.DataFrame(rows).head(sample_size)

def quick_qc(df, label):
    print(f"\nQC {label}: rows={len(df)} unique_doi={df['doi'].nunique() if len(df) else 0}")
    if len(df):
        if "repository_code" in df.columns:
            print("repo_code top:", df["repository_code"].value_counts().head(10).to_dict())
        if "dc_text_best_wc" in df.columns:
            print("best_wc median:", float(df["dc_text_best_wc"].median()))
            print("best_wc p10/p90:", float(df["dc_text_best_wc"].quantile(0.10)), float(df["dc_text_best_wc"].quantile(0.90)))
        if "dc_text_best_type" in df.columns:
            print("best_type:", df["dc_text_best_type"].value_counts().head(10).to_dict())
        if "has_any_keyword" in df.columns:
            print("has_any_keyword mean:", float(df["has_any_keyword"].mean()))

In [61]:
base = Path(".")
banned_union = load_banned_words_union(base)

In [62]:
df_B = scrape_B_federal_no_flagged_cursor(sample_size=2500, banned_union=banned_union)
quick_qc(df_B, "B")

[B_CURSOR] clients_chunk=1/2 page=1 added=151 total=151/2500 drops={'dupe': 0, 'no_rich_text': 9, 'has_kw': 222, 'too_short': 618}
[B_CURSOR] clients_chunk=1/2 page=2 added=801 total=952/2500 drops={'dupe': 0, 'no_rich_text': 14, 'has_kw': 268, 'too_short': 766}
[B_CURSOR] clients_chunk=1/2 page=3 added=970 total=1922/2500 drops={'dupe': 0, 'no_rich_text': 14, 'has_kw': 270, 'too_short': 794}
[B_CURSOR] clients_chunk=1/2 page=4 added=578 total=2500/2500 drops={'dupe': 0, 'no_rich_text': 14, 'has_kw': 270, 'too_short': 832}

QC B: rows=2500 unique_doi=2500
repo_code top: {'doe.lbnl': 2416, 'doe.ornl': 50, 'doe.nrel': 32, 'doe.osti': 2}
best_wc median: 122.0
best_wc p10/p90: 63.0 243.0
best_type: {'abstract': 2500}
has_any_keyword mean: 0.0


In [63]:
df_C = scrape_C_flagged_no_us_aff_cursor(sample_size=2500, banned_union=banned_union)
quick_qc(df_C, "C")

[C_CURSOR] kw_chunk=1/19 page=1 added=946 total=946/2500 drops={'dupe': 0, 'no_rich_text': 0, 'no_kw': 29, 'us_aff': 25}
[C_CURSOR] kw_chunk=1/19 page=2 added=966 total=1912/2500 drops={'dupe': 0, 'no_rich_text': 0, 'no_kw': 60, 'us_aff': 28}
[C_CURSOR] kw_chunk=1/19 page=3 added=588 total=2500/2500 drops={'dupe': 0, 'no_rich_text': 0, 'no_kw': 85, 'us_aff': 40}

QC C: rows=2500 unique_doi=2500
repo_code top: {'gesis.icpsr': 1269, 'dryad.dryad': 323, 'gesis.gesis': 276, 'cern.hepdata': 164, 'purdue.purduelib': 82, 'cdl.ucsb': 68, 'gbif.gbif': 64, 'pangaea.repository': 46, 'dans.archive': 28, 'bl.ads': 19}
best_wc median: 283.0
best_wc p10/p90: 142.9 771.0
best_type: {'abstract': 2103, 'other': 211, 'methods': 145, 'tableofcontents': 31, 'technicalinfo': 10}
has_any_keyword mean: 1.0


In [64]:
df_A = pd.read_csv('00_data/drp_withdoi.csv')

In [84]:
df_A["dc_publicationYear"] = pd.to_numeric(df_A["dc_publicationYear"], errors="coerce")

print("Min year:", df_A["dc_publicationYear"].min())
print("Max year:", df_A["dc_publicationYear"].max())
print("Median year:", df_A["dc_publicationYear"].median())
print(df_A["dc_publicationYear"].describe())

Min year: 1980.0
Max year: 2026.0
Median year: 2025.0
count    2539.000000
mean     2025.307995
std         1.105390
min      1980.000000
25%      2025.000000
50%      2025.000000
75%      2026.000000
max      2026.000000
Name: dc_publicationYear, dtype: float64


In [85]:
df_A["dc_publicationYear"].value_counts().sort_index().tail(20)

dc_publicationYear
1980.0       1
2017.0       2
2018.0       4
2019.0       1
2020.0       2
2024.0      10
2025.0    1622
2026.0     897
Name: count, dtype: int64

In [72]:
df_A["group"] = "A_drp"
df_B["group"] = "B_federal_no_flagged"
df_C["group"] = "C_flagged_no_us_aff"

df_A["label_vulnerable"] = 1
df_B["label_vulnerable"] = 0
df_C["label_vulnerable"] = 0

df_A["source"] = "drp"
df_B["source"] = "datacite"
df_C["source"] = "datacite"

df_BC = pd.concat([df_B, df_C], ignore_index=True, axis=0)

In [73]:
df_BC

,dataset_group,doi,repository_code,dc_language,dc_publicationYear,dc_url,dc_title,dc_types_resourceTypeGeneral,has_any_keyword,dc_publisher,...,dc_desc_seriesinformation,dc_desc_tableofcontents,dc_desc_technicalinfo,dc_text_best,dc_text_best_type,dc_text_best_wc,group,label_vulnerable,source,has_us_affiliation
0,B,10.3334/cdiac/atg.db1019,doe.lbnl,en,1997,https://www.osti.gov/servlets/purl/1389379/,The Environmental Measurements Laboratory's St...,Dataset,0,{'name': 'Environmental System Science Data In...,...,None,None,None,RANDAB represents the worlds largest collectio...,abstract,244,B_federal_no_flagged,0,datacite,NaN
1,B,10.3334/cdiac/lue.db1016,doe.lbnl,en,1996,https://www.osti.gov/servlets/purl/1389495/,"Global Population Distribution (1990),Terrestr...",Dataset,0,{'name': 'Environmental System Science Data In...,...,None,None,None,This data base contains gridded (one degree by...,abstract,171,B_federal_no_flagged,0,datacite,NaN
2,B,10.3334/cdiac/ffe.db1013.2011,doe.lbnl,en,1996,https://www.osti.gov/servlets/purl/1389444/,Annual Fossil-Fuel CO2 Emissions: Global Stabl...,Dataset,0,{'name': 'Environmental System Science Data In...,...,None,None,None,The 2011 revision of this database contains es...,abstract,116,B_federal_no_flagged,0,datacite,NaN
3,B,10.3334/cdiac/spruce.001,doe.ornl,en,2015,https://www.osti.gov/servlets/purl/1415768/,SPRUCE Environmental Monitoring Data: 2010-2016,Dataset,0,{'name': 'ORNLTESSFA (Oak Ridge National Lab's...,...,None,None,None,This data set reports selected ambient environ...,abstract,149,B_federal_no_flagged,0,datacite,NaN
4,B,10.3334/cdiac/amf.us-ton.m,doe.lbnl,en,2016,http://ameriflux.lbl.gov/doi/AmeriFlux/US-Ton,AmeriFlux Radiological and Meteorological Data...,Dataset,0,"{'name': 'University of California, Berkeley'}",...,None,None,None,Located in the lower foothills of the Sierra N...,abstract,148,B_federal_no_flagged,0,datacite,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,C,10.5061/dryad.cv791,dryad.dryad,en,2018,https://datadryad.org/dataset/doi:10.5061/drya...,Data from: Novel fine-scale aerial mapping app...,Dataset,1,"{'name': 'Dryad', 'schemeUri': 'https://ror.or...",...,None,None,None,Invasive weeds threaten the biodiversity and f...,abstract,289,C_flagged_no_us_aff,0,datacite,0.0
4996,C,10.5063/f1ns0rzx,cdl.ucsb,en,2017,https://knb.ecoinformatics.org/view/doi:10.506...,"Puerto Rico Bicknell's Thrush Surveys, 2015",Dataset,1,{'name': 'KNB Data Repository'},...,None,None,None,A predictive model of the distribution of wint...,abstract,294,C_flagged_no_us_aff,0,datacite,0.0
4997,C,10.5061/dryad.20864,dryad.dryad,en,2017,https://datadryad.org/dataset/doi:10.5061/drya...,Data from: Does detection range matter for inf...,Dataset,1,"{'name': 'Dryad', 'schemeUri': 'https://ror.or...",...,None,None,None,Accurately estimating contacts between animals...,abstract,264,C_flagged_no_us_aff,0,datacite,0.0
4998,C,10.5285/e5bfac9b-0642-4b5b-a780-e5801b2dab8b,bl.nerc,en,2017,https://catalogue.ceh.ac.uk/id/e5bfac9b-0642-4...,Source Attribution - deposition of nitrogen an...,Dataset,1,{'name': 'NERC Environmental Information Data ...,...,None,None,None,The FRAME (Fine Resolution Multi-pollutant Exc...,methods,264,C_flagged_no_us_aff,0,datacite,0.0


In [74]:
df_BC['dc_abstract'] = df_BC['dc_desc_abstract']

In [75]:
shared_cols = sorted(set(df_A.columns) & set(df_BC.columns))

print("Number of shared columns:", len(shared_cols))
print(shared_cols)

Number of shared columns: 12
['dc_abstract', 'dc_fundingReferences', 'dc_language', 'dc_publicationYear', 'dc_publisher', 'dc_title', 'dc_types_resourceTypeGeneral', 'dc_url', 'doi', 'group', 'label_vulnerable', 'source']


In [76]:
shared_cols = sorted(set(df_A.columns) & set(df_BC.columns))

df_A_shared = df_A[shared_cols].copy()
df_BC_shared = df_BC[shared_cols].copy()

df_all = pd.concat([df_A_shared, df_BC_shared], ignore_index=True)

print("Final stacked shape:", df_all.shape)

Final stacked shape: (7539, 12)


flagged words extraction

In [77]:
df_all

,dc_abstract,dc_fundingReferences,dc_language,dc_publicationYear,dc_publisher,dc_title,dc_types_resourceTypeGeneral,dc_url,doi,group,label_vulnerable,source
0,Section 10(j) of the National Labor Relations ...,[],en,2025.0,ICPSR - Interuniversity Consortium for Politic...,10(j) Injunctions,Dataset,https://www.datalumos.org/datalumos/project/22...,10.3886/e226824v1,A_drp,1,drp
1,CDC monitors invasive bacterial infections tha...,[],en,2026.0,ICPSR - Interuniversity Consortium for Politic...,1998-2023 Serotype Data for Invasive Pneumococ...,Dataset,https://www.datalumos.org/datalumos/project/24...,10.3886/E243434V1,A_drp,1,drp
2,This dataset contains public information on gr...,[],en,2025.0,ICPSR - Interuniversity Consortium for Politic...,2022-2024 MBDA Grantees,Dataset,https://www.datalumos.org/datalumos/project/22...,10.3886/E223443V1,A_drp,1,drp
3,This submission includes publicly available da...,[],NaN,2025.0,Harvard Dataverse,Extracted Data From: Downloadable 2006 IUR Pub...,Dataset,https://dataverse.harvard.edu/citation?persist...,10.7910/dvn/f3a62w,A_drp,1,drp
4,The map illustrates the total number of 2013 a...,[],en,2026.0,ICPSR - Interuniversity Consortium for Politic...,2013-2014 PHAP Associates by State,Dataset,https://www.datalumos.org/datalumos/project/24...,10.3886/E244041V1,A_drp,1,drp
...,...,...,...,...,...,...,...,...,...,...,...,...
7534,Invasive weeds threaten the biodiversity and f...,[],en,2018.0,"{'name': 'Dryad', 'schemeUri': 'https://ror.or...",Data from: Novel fine-scale aerial mapping app...,Dataset,https://datadryad.org/dataset/doi:10.5061/drya...,10.5061/dryad.cv791,C_flagged_no_us_aff,0,datacite
7535,A predictive model of the distribution of wint...,[],en,2017.0,{'name': 'KNB Data Repository'},"Puerto Rico Bicknell's Thrush Surveys, 2015",Dataset,https://knb.ecoinformatics.org/view/doi:10.506...,10.5063/f1ns0rzx,C_flagged_no_us_aff,0,datacite
7536,Accurately estimating contacts between animals...,[],en,2017.0,"{'name': 'Dryad', 'schemeUri': 'https://ror.or...",Data from: Does detection range matter for inf...,Dataset,https://datadryad.org/dataset/doi:10.5061/drya...,10.5061/dryad.20864,C_flagged_no_us_aff,0,datacite
7537,This dataset contains 90 source footprints of ...,[],en,2017.0,{'name': 'NERC Environmental Information Data ...,Source Attribution - deposition of nitrogen an...,Dataset,https://catalogue.ceh.ac.uk/id/e5bfac9b-0642-4...,10.5285/e5bfac9b-0642-4b5b-a780-e5801b2dab8b,C_flagged_no_us_aff,0,datacite


In [78]:
df_all = add_flagged_column(df_all, abstract_col='dc_abstract')

Cleaning

clean publisher

In [ ]:
def clean_publisher(val):
    if pd.isna(val):
        return None

    if isinstance(val, dict):
        return val.get("name")

    if isinstance(val, str) and val.strip().startswith("{"):
        try:
            parsed = ast.literal_eval(val)
            if isinstance(parsed, dict):
                return parsed.get("name")
        except:
            return val

    return val

df_all["dc_publisher"] = df_all["dc_publisher"].apply(clean_publisher)

In [80]:
df_all['dc_fundingReferences'].value_counts()

dc_fundingReferences
[]                                                                                                                                                                                                                                                                                       4467
[]                                                                                                                                                                                                                                                                                       2492
[{'funderName': 'United States Department of Justice. Office of Justice Programs. National Institute of Justice'}]                                                                                                                                                                         71
[{'funderName': 'National Science Foundation'}]                                                                          

In [ ]:
def extract_first_funder(val):

    # Handle None / NaN safely
    if val is None:
        return None

    # If numpy array → convert to list
    if isinstance(val, np.ndarray):
        val = val.tolist()

    # If stringified list → parse
    if isinstance(val, str):
        val = val.strip()
        if not val or val == "[]":
            return None
        try:
            val = ast.literal_eval(val)
        except:
            return None

    # Now expect list
    if isinstance(val, list) and len(val) > 0:
        first = val[0]
        if isinstance(first, dict):
            return first.get("funderName")

    return None

df_all["dc_funder_first"] = df_all["dc_fundingReferences"].apply(extract_first_funder)

In [82]:
df_all["dc_funder_first"].value_counts().head(20)

dc_funder_first
United States Department of Justice. Office of Justice Programs. National Institute of Justice                                                                           180
National Science Foundation                                                                                                                                              120
United States Department of the Interior. United States Geological Survey                                                                                                 20
United States Department of Health and Human Services. Substance Abuse and Mental Health Services Administration. Center for Behavioral Health Statistics and Quality     20
United States Department of Health and Human Services. Substance Abuse and Mental Health Services Administration. Office of Applied Studies                               17
Robert Wood Johnson Foundation                                                                                         

In [90]:
dupes = df_all[df_all.duplicated(subset=["doi", "dc_abstract"], keep=False)]

print("Number of duplicate rows:", len(dupes))
dupes.sort_values(["doi", "dc_abstract"]).head(20)

Number of duplicate rows: 300


,dc_abstract,dc_fundingReferences,dc_language,dc_publicationYear,dc_publisher,dc_title,dc_types_resourceTypeGeneral,dc_url,doi,group,label_vulnerable,source,flagged_words_pen,flagged_words_nyt,flagged_words_all,has_flagged_word,num_flagged_words,num_pen_words,num_nyt_words,dc_funder_first
431,The Declaration Denials dataset lists all requ...,[],en,2025.0,ICPSR - Interuniversity Consortium for Politic...,FEMA Disaster Information,Dataset,https://www.datalumos.org/datalumos/project/21...,10.3886/E218462V1,A_drp,1,drp,tribal,status; tribal,status; tribal,1,2,1,2,None
446,The Declaration Denials dataset lists all requ...,[],en,2025.0,ICPSR - Interuniversity Consortium for Politic...,FEMA Disaster Information,Dataset,https://www.datalumos.org/datalumos/project/21...,10.3886/E218462V1,A_drp,1,drp,tribal,status; tribal,status; tribal,1,2,1,2,None
594,The Declaration Denials dataset lists all requ...,[],en,2025.0,ICPSR - Interuniversity Consortium for Politic...,FEMA Disaster Information,Dataset,https://www.datalumos.org/datalumos/project/21...,10.3886/E218462V1,A_drp,1,drp,tribal,status; tribal,status; tribal,1,2,1,2,None
595,The Declaration Denials dataset lists all requ...,[],en,2025.0,ICPSR - Interuniversity Consortium for Politic...,FEMA Disaster Information,Dataset,https://www.datalumos.org/datalumos/project/21...,10.3886/E218462V1,A_drp,1,drp,tribal,status; tribal,status; tribal,1,2,1,2,None
596,The Declaration Denials dataset lists all requ...,[],en,2025.0,ICPSR - Interuniversity Consortium for Politic...,FEMA Disaster Information,Dataset,https://www.datalumos.org/datalumos/project/21...,10.3886/E218462V1,A_drp,1,drp,tribal,status; tribal,status; tribal,1,2,1,2,None
1422,The Declaration Denials dataset lists all requ...,[],en,2025.0,ICPSR - Interuniversity Consortium for Politic...,FEMA Disaster Information,Dataset,https://www.datalumos.org/datalumos/project/21...,10.3886/E218462V1,A_drp,1,drp,tribal,status; tribal,status; tribal,1,2,1,2,None
1170,The Housing Assistance Program Data Owners dat...,[],en,2025.0,ICPSR - Interuniversity Consortium for Politic...,FEMA Disaster Individual Assistance,Dataset,https://www.datalumos.org/datalumos/project/21...,10.3886/E218466V1,A_drp,1,drp,,status,status,1,1,0,1,None
1171,The Housing Assistance Program Data Owners dat...,[],en,2025.0,ICPSR - Interuniversity Consortium for Politic...,FEMA Disaster Individual Assistance,Dataset,https://www.datalumos.org/datalumos/project/21...,10.3886/E218466V1,A_drp,1,drp,,status,status,1,1,0,1,None
1207,The Housing Assistance Program Data Owners dat...,[],en,2025.0,ICPSR - Interuniversity Consortium for Politic...,FEMA Disaster Individual Assistance,Dataset,https://www.datalumos.org/datalumos/project/21...,10.3886/E218466V1,A_drp,1,drp,,status,status,1,1,0,1,None
1208,The Housing Assistance Program Data Owners dat...,[],en,2025.0,ICPSR - Interuniversity Consortium for Politic...,FEMA Disaster Individual Assistance,Dataset,https://www.datalumos.org/datalumos/project/21...,10.3886/E218466V1,A_drp,1,drp,,status,status,1,1,0,1,None


In [91]:
df_all = df_all.drop_duplicates(subset=["doi", "dc_abstract"]).copy()

Matching distributions

In [5]:
df_all['abstract_len'] = df_all['dc_abstract'].fillna("").str.split().str.len()
df_all['title_len'] = df_all['dc_title'].fillna("").str.split().str.len()

df_all[['abstract_len', 'title_len']].describe()

,abstract_len,title_len
count,7272.000000,7272.000000
mean,219.387239,9.159653
std,269.087670,4.757344
min,0.000000,1.000000
25%,79.000000,7.000000
50%,145.000000,7.000000
75%,261.000000,11.000000
max,9262.000000,40.000000


In [6]:
upper_cap = df_all["abstract_len"].quantile(0.95)
df_all = df_all[df_all["abstract_len"] <= upper_cap]

In [7]:
df_matched = df_all[
    (df_all["abstract_len"] >= 75) &
    (df_all["abstract_len"] <= 160)
]

In [9]:
df_all[['abstract_len', 'title_len']].describe()

,abstract_len,title_len
count,6909.000000,6909.000000
mean,176.541178,9.130120
std,133.006383,4.766858
min,0.000000,1.000000
25%,75.000000,7.000000
50%,139.000000,7.000000
75%,244.000000,10.000000
max,652.000000,40.000000


In [10]:
df_all.to_csv('00_data/df_all.csv')

In [4]:
df_all = pd.read_csv('00_data/df_all.csv')